## Model training

Six models (Logistic Regression, Random Forest, XGBoost, LightGBM, EBM and TabPFN) are compared on each age group using repeated stratified 5-fold cross-validation with 3 repeats. Hyperparameters are tuned inside each training fold, SMOTENC is applied to the training data only, and the score column is not used. The notebook also runs the cross-source test (RQ2) and saves the final model for each age group for the web app.

The full run is slow, mostly because of EBM and TabPFN. Set `RERUN = False` to load the saved results instead.

In [1]:
# TabPFN (PyTorch) has to be imported before LightGBM/XGBoost, otherwise the
# two OpenMP runtimes can clash and crash the kernel.
from tabpfn import TabPFNClassifier

import shutil
import time
from pathlib import Path

import joblib
import pandas as pd
from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline
from interpret.glassbox import ExplainableBoostingClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, brier_score_loss, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

PROCESSED = Path("../data/processed")
TABLES = Path("../results/tables")
MODELS = Path("../results/models")
APP = Path("../app")
for p in [TABLES, MODELS, APP / "models", APP / "data"]:
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
RERUN = True

GROUPS = ["toddler", "child", "adolescent", "adult"]
QUESTIONS = [f"A{i}" for i in range(1, 11)]
FEATURES = QUESTIONS + ["age", "male", "jaundice", "family_asd"]

data = {name: pd.read_csv(PROCESSED / f"{name}.csv") for name in GROUPS + ["toddler_saudi", "adult_kaggle"]}

### 1. Models and parameter grids

The grids are kept small on purpose: the datasets are tiny and most of these models work well with their defaults.

In [2]:
MODEL_NAMES = ["Logistic Regression", "Random Forest", "XGBoost", "LightGBM", "EBM", "TabPFN"]


def get_classifier(name):
    if name == "Logistic Regression":
        return LogisticRegression(max_iter=2000), {"C": [0.01, 0.1, 1, 10]}
    if name == "Random Forest":
        return (RandomForestClassifier(random_state=SEED, n_jobs=1),
                {"n_estimators": [200, 500], "max_depth": [None, 5, 10]})
    if name == "XGBoost":
        return (XGBClassifier(random_state=SEED, n_jobs=1, eval_metric="logloss"),
                {"n_estimators": [100, 300], "max_depth": [3, 5], "learning_rate": [0.05, 0.1]})
    if name == "LightGBM":
        return (LGBMClassifier(random_state=SEED, n_jobs=1, min_child_samples=10, verbose=-1),
                {"n_estimators": [100, 300], "num_leaves": [7, 15], "learning_rate": [0.05, 0.1]})
    if name == "EBM":
        return (ExplainableBoostingClassifier(random_state=SEED, n_jobs=1, outer_bags=8),
                {"interactions": [0, 10]})
    if name == "TabPFN":
        # after SMOTE the toddler training folds pass 1,000 rows; TabPFN v2 supports
        # up to 10,000 but blocks >1,000 on CPU by default only because it gets slow
        return TabPFNClassifier(device="cpu", random_state=SEED, ignore_pretraining_limits=True), {}


def make_pipeline(name, features):
    clf, grid = get_classifier(name)
    categorical = [i for i, f in enumerate(features) if f != "age"]
    steps = [("smote", SMOTENC(categorical_features=categorical, random_state=SEED))]
    if name == "Logistic Regression":
        steps.append(("scale", StandardScaler()))
    steps.append(("clf", clf))
    return Pipeline(steps), {f"clf__{k}": v for k, v in grid.items()}

### 2. Training and scoring helpers

`fit` tunes a model on the data it is given and nothing else. `scores` computes the metrics from the proposal plus the **Brier score**, which checks whether the predicted probabilities are trustworthy and not just the yes/no decisions.

In [3]:
def fit(name, X, y, features):
    model, grid = make_pipeline(name, features)
    if not grid:  # TabPFN
        return model.fit(X, y), {}
    inner = StratifiedKFold(3, shuffle=True, random_state=SEED)
    search = GridSearchCV(model, grid, cv=inner, scoring="roc_auc", n_jobs=-1)
    search.fit(X, y)
    return search.best_estimator_, search.best_params_


def scores(y_true, prob):
    pred = (prob >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, prob),
        "brier": brier_score_loss(y_true, prob),
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
    }

### 3. Nested cross-validation (RQ1)

In [4]:
def nested_cv(name, df, features):
    X, y = df[features], df["asd"]
    outer = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
    rows, preds = [], []
    for fold, (train, test) in enumerate(outer.split(X, y)):
        model, params = fit(name, X.iloc[train], y.iloc[train], features)
        prob = model.predict_proba(X.iloc[test])[:, 1]
        rows.append({"model": name, "fold": fold, **scores(y.iloc[test], prob), "params": str(params)})
        # out-of-fold predictions, used later for the bias check
        preds.append(pd.DataFrame({"model": name, "repeat": fold // 5, "row": test, "prob": prob}))
    return pd.DataFrame(rows), pd.concat(preds)

In [5]:
if RERUN or not (TABLES / "cv_folds.csv").exists():
    results, predictions = [], []
    for g in GROUPS:
        for name in MODEL_NAMES:
            start = time.time()
            res, pred = nested_cv(name, data[g], FEATURES)
            res.insert(0, "group", g)
            pred.insert(0, "group", g)
            results.append(res)
            predictions.append(pred)
            print(f"{g:<11} {name:<20} AUC {res.roc_auc.mean():.3f}  recall {res.recall.mean():.3f}"
                  f"  ({time.time() - start:.0f}s)")
    cv = pd.concat(results, ignore_index=True)
    cv.to_csv(TABLES / "cv_folds.csv", index=False)
    pd.concat(predictions).to_csv(TABLES / "cv_predictions.csv", index=False)
else:
    cv = pd.read_csv(TABLES / "cv_folds.csv")

toddler     Logistic Regression  AUC 1.000  recall 0.999  (9s)


toddler     Random Forest        AUC 0.995  recall 0.978  (22s)


toddler     XGBoost              AUC 0.999  recall 0.987  (5s)


toddler     LightGBM             AUC 0.998  recall 0.980  (5s)


toddler     EBM                  AUC 1.000  recall 1.000  (559s)


toddler     TabPFN               AUC 1.000  recall 1.000  (249s)


... (14 outputs left out)

adult       XGBoost              AUC 0.998  recall 0.977  (4s)


adult       LightGBM             AUC 0.997  recall 0.930  (3s)


adult       EBM                  AUC 1.000  recall 1.000  (527s)


adult       TabPFN               AUC 1.000  recall 1.000  (140s)


In [6]:
cv.groupby(["group", "model"], sort=False)[["accuracy", "recall", "f1", "roc_auc", "brier"]].mean().round(3)

accuracy  recall     f1  roc_auc  brier
group      model                                                       
toddler    Logistic Regression     0.999   0.999  0.999    1.000  0.008
           Random Forest           0.962   0.978  0.973    0.995  0.035
           XGBoost                 0.986   0.987  0.990    0.999  0.013
           LightGBM                0.977   0.980  0.984    0.998  0.018
           EBM                     1.000   1.000  1.000    1.000  0.000
           TabPFN                  1.000   1.000  1.000    1.000  0.000
child      Logistic Regression     0.999   1.000  0.999    1.000  0.028
           Random Forest           0.938   0.955  0.937    0.988  0.075
           XGBoost                 0.949   0.967  0.948    0.992  0.037
           LightGBM                0.949   0.952  0.947    0.990  0.043
           EBM                     1.000   1.000  1.000    1.000  0.000
           TabPFN                  0.987   0.990  0.986    1.000  0.011
adolescent Logistic Regression     0.918   0.972  0.934    0.988  0.073
           Random Forest           0.892   0.956  0.913    0.977  0.085
           XGBoost                 0.878   0.939  0.901    0.965  0.081
           LightGBM                0.888   0.939  0.908    0.964  0.089
           EBM                     0.937   0.967  0.949    0.986  0.051
           TabPFN                  0.931   0.972  0.945    0.990  0.047
adult      Logistic Regression     1.000   1.000  1.000    1.000  0.004
           Random Forest           0.972   0.941  0.947    0.996  0.034
           XGBoost                 0.981   0.977  0.966    0.998  0.015
           LightGBM                0.968   0.930  0.940    0.997  0.023
           EBM                     1.000   1.000  1.000    1.000  0.000
           TabPFN                  1.000   1.000  1.000    1.000  0.001

### 4. Cross-source test (RQ2)

Each model is trained on one source and tested on the other, in both directions:

* Thabtah toddlers ↔ Saudi toddlers (both Q-CHAT-10)
* UCI adults ↔ Kaggle adults (both AQ-10 Adult)

The Saudi data has no jaundice column, so `jaundice` is left out of this experiment for all pairs.

In [7]:
PAIRS = [("toddler", "toddler_saudi"), ("toddler_saudi", "toddler"),
         ("adult", "adult_kaggle"), ("adult_kaggle", "adult")]
CROSS_FEATURES = [f for f in FEATURES if f != "jaundice"]

if RERUN or not (TABLES / "cross_source.csv").exists():
    rows = []
    for train_name, test_name in PAIRS:
        for name in MODEL_NAMES:
            model, _ = fit(name, data[train_name][CROSS_FEATURES], data[train_name]["asd"], CROSS_FEATURES)
            prob = model.predict_proba(data[test_name][CROSS_FEATURES])[:, 1]
            rows.append({"train": train_name, "test": test_name, "model": name,
                         **scores(data[test_name]["asd"], prob)})
        print("done", train_name, "->", test_name)
    cross = pd.DataFrame(rows)
    cross.to_csv(TABLES / "cross_source.csv", index=False)
else:
    cross = pd.read_csv(TABLES / "cross_source.csv")

cross.pivot_table(index=["train", "test"], columns="model", values="roc_auc", sort=False).round(3)

done toddler -> toddler_saudi


done toddler_saudi -> toddler


done adult -> adult_kaggle


/Users/muhammadusman/Desktop/finalthesis sep 2026/hamza/asd-screening/.venv/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


done adult_kaggle -> adult


,model,Logistic Regression,Random Forest,XGBoost,LightGBM,EBM,TabPFN
train,test,,,,,,
toddler,toddler_saudi,1.000,0.998,1.000,1.000,1.000,1.000
toddler_saudi,toddler,1.000,0.986,0.994,0.992,1.000,1.000
adult,adult_kaggle,0.902,0.896,0.903,0.898,0.879,0.871
adult_kaggle,adult,0.998,0.970,0.954,0.951,0.984,0.952


### 5. Final model for the web app

For each age group the model with the best mean ROC-AUC is chosen (recall breaks ties, since missing a true case is the worst error in screening). If two models are level, the more interpretable one wins.

TabPFN is left out of this choice: it needs PyTorch and a 100 MB+ weights file, which does not fit comfortably in the free Streamlit Cloud tier. Its results are still reported in the paper.

The chosen model is then retrained on **all** data of that group and saved together with a small background sample, which the app uses to compute SHAP explanations.

In [8]:
PREFERENCE = ["EBM", "Logistic Regression", "Random Forest", "LightGBM", "XGBoost"]

summary = cv.groupby(["group", "model"])[["roc_auc", "recall", "f1", "accuracy"]].mean().reset_index()
summary = summary[summary.model.isin(PREFERENCE)].copy()
summary["roc_auc_r"] = summary["roc_auc"].round(3)
summary["recall_r"] = summary["recall"].round(3)
summary["preference"] = summary["model"].map(PREFERENCE.index)

best = (summary.sort_values(["roc_auc_r", "recall_r", "preference"], ascending=[False, False, True])
               .groupby("group").head(1).set_index("group").loc[GROUPS])
best[["model", "roc_auc", "recall", "f1"]]

,model,roc_auc,recall,f1
group,,,,
toddler,EBM,1.000000,1.000000,1.000000
child,EBM,1.000000,1.000000,1.000000
adolescent,Logistic Regression,0.987731,0.972222,0.934363
adult,EBM,1.000000,1.000000,1.000000


In [9]:
for g in GROUPS:
    name = best.loc[g, "model"]
    df = data[g]
    model, params = fit(name, df[FEATURES], df["asd"], FEATURES)
    bundle = {
        "group": g,
        "model_name": name,
        "model": model,
        "features": FEATURES,
        "params": params,
        "background": df[FEATURES].sample(min(100, len(df)), random_state=SEED),
    }
    joblib.dump(bundle, MODELS / f"{g}.joblib")
    shutil.copy(MODELS / f"{g}.joblib", APP / "models" / f"{g}.joblib")
    print(f"{g:<11} {name:<20} {params}")

toddler     EBM                  {'clf__interactions': 0}


child       EBM                  {'clf__interactions': 0}


adolescent  Logistic Regression  {'clf__C': 1}


adult       EBM                  {'clf__interactions': 0}


The app also shows the evaluation results, so the summary tables are copied next to it.

In [10]:
metrics = ["accuracy", "precision", "recall", "f1", "roc_auc", "brier"]
cv.groupby(["group", "model"], sort=False)[metrics].agg(["mean", "std"]).round(4).to_csv(APP / "data" / "cv_summary.csv")
cross.to_csv(APP / "data" / "cross_source.csv", index=False)
best[["model", "roc_auc", "recall", "f1", "accuracy"]].to_csv(APP / "data" / "chosen_models.csv")